In [ ]:
import os
import sys
import yaml
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sys.path.insert(0, '../src')
np.random.seed(42)

from data_loader import DataLoader
from model_trainer import ModelEvaluator
from retraining_strategy import (
    NoRetrainingStrategy, PeriodicRetrainingStrategy,
    TriggerBasedRetrainingStrategy, SlidingWindowRetrainingStrategy,
    StrategyExecutor
)

# Prepare data
with open('../config.yaml', 'r') as f:
    config = yaml.safe_load(f)

loader = DataLoader(random_state=42)
df_synthetic, _ = loader.load_synthetic_data(
    n_samples_per_window=config['dataset']['synthetic']['n_samples_per_window'],
    n_features=config['dataset']['synthetic']['n_features'],
    n_windows=config['dataset']['synthetic']['n_windows'],
    drift_magnitude=config['dataset']['synthetic']['drift_magnitude'],
    drift_type=config['dataset']['synthetic']['drift_type']
)

windows = loader.split_into_windows(df_synthetic, n_windows=config['time_window']['n_windows'])
splits = loader.get_ref_and_test_splits(windows, ref_window_idx=0)
splits = loader.standardize_splits(splits)

print(f"✓ Data prepared: {len(splits)} windows")


In [ ]:
# Initialize evaluator and executor
evaluator = ModelEvaluator(metrics=['accuracy', 'f1', 'auc_roc', 'brier_score'])
executor = StrategyExecutor(evaluator=evaluator)

# Define retraining strategies
strategies = {
    'no_retrain': NoRetrainingStrategy(model_type='random_forest'),
    'periodic_k1': PeriodicRetrainingStrategy(period=1, model_type='random_forest'),
    'periodic_k2': PeriodicRetrainingStrategy(period=2, model_type='random_forest'),
    'periodic_k3': PeriodicRetrainingStrategy(period=3, model_type='random_forest'),
    'sliding_w1': SlidingWindowRetrainingStrategy(window_size=1, model_type='random_forest'),
    'sliding_w2': SlidingWindowRetrainingStrategy(window_size=2, model_type='random_forest'),
}

print("Executing retraining strategies...")
strategy_results = executor.execute_multiple_strategies(strategies, splits)

print("\n✓ All strategies executed")
for strategy_name, (perf_df, retrain_windows) in strategy_results.items():
    print(f"{strategy_name}: {len(retrain_windows)} retrainings")


In [ ]:
# Compare strategies
comparison_rows = []
for strategy_name, (perf_df, retrain_windows) in strategy_results.items():
    row = {
        'strategy': strategy_name,
        'n_retrains': len(retrain_windows),
        'avg_auc': perf_df['auc_roc'].mean(),
        'avg_f1': perf_df['f1'].mean(),
        'min_auc': perf_df['auc_roc'].min(),
        'min_f1': perf_df['f1'].min(),
    }
    comparison_rows.append(row)

comparison_df = pd.DataFrame(comparison_rows)
print("\nStrategy Comparison:")
print(comparison_df)

# Save comparison
comparison_df.to_csv('../reports/strategy_comparison.csv', index=False)
print("\n✓ Comparison saved to CSV")


In [ ]:
# Plot strategy comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# AUC-ROC comparison
ax = axes[0]
for strategy_name, (perf_df, _) in strategy_results.items():
    ax.plot(perf_df['window'], perf_df['auc_roc'], marker='o', label=strategy_name, linewidth=2)
ax.set_xlabel("Time Window")
ax.set_ylabel("AUC-ROC")
ax.set_title("AUC-ROC Across Strategies")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# Retraining cost vs. performance
ax = axes[1]
for strategy_name, (perf_df, _) in strategy_results.items():
    n_retrains = len([s for s in strategy_results[strategy_name][1]])
    avg_auc = perf_df['auc_roc'].mean()
    ax.scatter(n_retrains, avg_auc, s=200, alpha=0.7, label=strategy_name)
ax.set_xlabel("Number of Retrainings")
ax.set_ylabel("Average AUC-ROC")
ax.set_title("Retraining Cost vs. Performance")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../reports/figures/04_strategy_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Strategy comparison plot saved")


## 4. Summary: Strategy Effectiveness

**Key Findings from Strategy Comparison:**
- Periodic retraining (K=1) achieves the highest performance but requires frequent updates
- Sliding window retraining provides good performance with moderate cost
- No retraining baseline shows significant performance degradation
- Trade-off between computational cost (number of retrains) and model performance exists


## 3. Visualize Strategy Performance


## 2. Compare Strategy Performance


## 1. Create and Execute Retraining Strategies


# 04_retraining_strategies: Implementing and Comparing Retraining Strategies

This notebook implements multiple retraining strategies and compares their effectiveness in maintaining model performance.
